# 07 · MoE 稀疏混合专家 —— 医院大会诊，每次只进 2 个医生

**家族位置**：08 生产级优化第 7 站。06 是静态剪枝（一次剪完），本章是动态剪枝：8 个专家值班，每个 token 只叫 top-2，用 1/4 算力跑 4× 参数。

**学习目标**：top-k 门控稀疏激活；负载均衡 aux loss；MoE vs dense 同预算对照；专家分化可视化。

## 1. 原理：会诊只进相关科室

### 通俗理解

**一句话**：dense 是全院医生围着每个病人转（128 个全上）；MoE 是分诊台先看一眼，只叫最相关的 2 个科室（8 选 2）——病人多了不断招人（扩专家），单个病人的花费不涨（算力恒定）。

### 结构账

```
门诊： E=8 专家（小 MLP 128→256→128），top-k=2，门控 softmax 取 top2 重归一
均衡： aux=E·Σ(mean(g)·mean(hard))，防赢家通吃（λ=0.01）
对照： MoE 总参≈3× Dense，但激活参数同量级（MoE激活≈0.40M≈Dense 0.40M）；FLOPs MoE≈k/E=1/4
任务： 模加 S=16（相对推理，需容量）3000 条 30ep；另复制冷启动对照见 FAQ
```

In [ ]:
import sys, time
from pathlib import Path
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader
ROOT=Path.cwd()
while ROOT != ROOT.parent and not (ROOT/'common').exists(): ROOT=ROOT.parent
sys.path.insert(0,str(ROOT))
from common.data import make_modadd_data
from common.models import MoEGPT, DenseGPT
from common.utils import set_seed,setup_chinese_font,count_params
set_seed(0); setup_chinese_font()
FIGS=Path.cwd()/'figs'; FIGS.mkdir(exist_ok=True)
print('torch:',torch.__version__)
Xm,ym=make_modadd_data(3000,16,16,seed=2); Xn,yn=make_modadd_data(500,16,16,seed=3)
tr=DataLoader(TensorDataset(Xm,ym),batch_size=128,shuffle=True); va=DataLoader(TensorDataset(Xn,yn),batch_size=512)
print(f'modadd train {tuple(Xm.shape)} / val {tuple(Xn.shape)}')
torch.manual_seed(0)
moe=MoEGPT(vocab=16,dim=128,depth=2,heads=4,num_experts=8,top_k=2,expert_hidden=256)
torch.manual_seed(0)
dense=DenseGPT(vocab=16,dim=128,depth=2,heads=4,hidden=512)
print(f'MoE总参={count_params(moe)} | Dense总参={count_params(dense)} | 激活比≈k/E=2/8=1/4')

## 2. 同预算训练：MoE vs Dense 30ep

In [ ]:
import torch.nn as nn
def fit_m(m,epochs=30,lr=3e-3,aux_w=0.01,tag=''):
    opt=torch.optim.Adam(m.parameters(),lr=lr); crit=nn.CrossEntropyLoss(); hist=[]
    for ep in range(1,epochs+1):
        m.train(); tot=0
        for src,tgt in tr:
            logits=m(src); loss=crit(logits.reshape(-1,16),tgt.reshape(-1))
            if hasattr(m,'aux_loss'): loss=loss+aux_w*m.aux_loss().to(loss.device)
            opt.zero_grad(); loss.backward(); opt.step(); tot+=loss.item()*len(src)
        hist.append(tot/len(tr.dataset))
        if ep in (1,5,10,15,20,25,30):
            m.eval();
            with torch.no_grad():
                ok=sum(((m(s).argmax(-1)==t).all(1).sum().item()) for s,t in va); n=sum(len(s) for s,t in va)
            print(f'{tag} ep {ep:02d} loss {hist[-1]:.3f} val-seq {ok/n:.4f}',flush=True)
    return hist
h_moe=fit_m(moe,tag='MoE ')
h_den=fit_m(dense,tag='Dense')
fig,ax=plt.subplots(figsize=(6,3.2))
ax.plot(h_moe,label='MoE-8top2',color='#4C72B0'); ax.plot(h_den,label='Dense',color='#DD8452')
ax.set_xlabel('epoch'); ax.set_ylabel('CE loss'); ax.legend()
ax.set_title('同预算训练：MoE 激活 1/4 能跟上吗')
plt.tight_layout(); plt.savefig(FIGS/'fig1_train.png',dpi=150,bbox_inches='tight'); plt.show()

## 3. 稀疏红利：FLOPs/参数对照 + 专家分化

In [ ]:
moe.eval(); dense.eval()
with torch.no_grad():
    ok_m=sum(((moe(s).argmax(-1)==t).all(1).sum().item()) for s,t in va); n=sum(len(s) for s,t in va)
    ok_d=sum(((dense(s).argmax(-1)==t).all(1).sum().item()) for s,t in va)
print(f'MoE val-seq={ok_m/n:.4f} | Dense val-seq={ok_d/n:.4f}',flush=True)
print(f'总参 MoE={count_params(moe)} Dense={count_params(dense)}（总参MoE≈3×，激活参数同量级）| 激活FLOPs比≈1/4',flush=True)
probes=torch.randint(0,16,(256,16))
with torch.no_grad():
    moe.eval(); use=np.zeros(8)
    x=moe.pos(moe.emb(probes))
    for blk in moe.blocks:
        g=torch.softmax(blk.moe.gate(blk.n2(blk.attn(blk.n1(x))+x)),dim=-1)
        top=g.topk(2,-1).indices.flatten().numpy()
        for e in top: use[e]+=1
        x=blk(x)
    use=use/use.sum()
print('专家使用率:',np.round(use,3),flush=True)
fig,ax=plt.subplots(1,2,figsize=(9,3.2))
ax[0].bar(['MoE总参','Dense总参'],[count_params(moe),count_params(dense)],color=['#4C72B0','#DD8452'])
for i,v in enumerate([count_params(moe),count_params(dense)]): ax[0].text(i,v+5000,f'{v}',ha='center',fontsize=9)
ax[0].set_title('总参MoE≈3×，激活参数同量级')
ax[1].bar(range(8),use,color='#55A868')
ax[1].axhline(1/8,color='red',ls='--',label='均匀线 0.125')
ax[1].set_xlabel('expert'); ax[1].set_title('专家使用率（aux 均衡效果）'); ax[1].legend()
plt.tight_layout(); plt.savefig(FIGS/'fig2_experts.png',dpi=150,bbox_inches='tight'); plt.show()
fig,ax=plt.subplots(figsize=(6,2.8)); ax.axis('off')
ax.text(0.02,0.7,f'MoE val-seq={ok_m/n:.4f} vs Dense={ok_d/n:.4f}（总参3×，激活同量级，FLOPs 1/4）',fontsize=11)
ax.text(0.02,0.4,'门诊账本：8 专家值班，单 token 只花 2 个的钱',fontsize=11)
ax.text(0.02,0.1,'aux loss 让 8 个科室雨露均沾（红线=均匀），防一个科室累死',fontsize=11,color='#1a6b3c')
ax.set_title('08-07 总览')
plt.tight_layout(); plt.savefig(FIGS/'fig3_summary.png',dpi=150,bbox_inches='tight'); plt.show()
print('SUMMARY',round(ok_m/n,4),round(ok_d/n,4),count_params(moe),count_params(dense),np.round(use,3).tolist())